In [8]:
!pip install -q -U langchain langchain-community langchain-text-splitters pymupdf langchain-openai langchain_huggingface

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf")
data = loader.load()


In [2]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)
chunks = text_splitter.split_documents(data)

In [3]:
# Display 3 representative chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i+1} ---\n{chunk.page_content}\n")

--- Chunk 1 ---
ImageNet Classiﬁcation with Deep Convolutional
Neural Networks
Alex Krizhevsky
University of Toronto
kriz@cs.utoronto.ca
Ilya Sutskever
University of Toronto
ilya@cs.utoronto.ca
Geoffrey E. Hinton
University of Toronto
hinton@cs.utoronto.ca
Abstract
We trained a large, deep convolutional neural network to classify the 1.2 million
high-resolution images in the ImageNet LSVRC-2010 contest into the 1000 dif-
ferent classes. On the test data, we achieved top-1 and top-5 error rates of 37.5%

--- Chunk 2 ---
ferent classes. On the test data, we achieved top-1 and top-5 error rates of 37.5%
and 17.0% which is considerably better than the previous state-of-the-art. The
neural network, which has 60 million parameters and 650,000 neurons, consists
of ﬁve convolutional layers, some of which are followed by max-pooling layers,
and three fully-connected layers with a ﬁnal 1000-way softmax. To make train-
ing faster, we used non-saturating neurons and a very efﬁcient GPU implemen-



In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Setup Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in FAISS
vector_db = FAISS.from_documents(chunks, embeddings)

# Semantic Search Query
query = "What is the overall goal of the ImageNet classification experiment?"
docs = vector_db.similarity_search(query, k=3)

# Display results
for doc in docs:
    print(f"Result: {doc.page_content[:200]}...")

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# Initialize the LLM
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Actual token removed for submisison
your_hf_token = "..."

# Using open-source model
llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=your_hf_token,
    max_new_tokens=512
)

llm = ChatHuggingFace(llm=llm_endpoint)

# Define the Prompt
contextual_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Use the following context from the AlexNet paper to answer: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# Retrieval Logic
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

setup_and_retrieval = RunnableParallel({
    "context": (lambda x: x["question"]) | retriever | format_docs,
    "question": lambda x: x["question"],
    "chat_history": lambda x: x["chat_history"]
})

# Construct the LCEL Chain
rag_chain = (
    setup_and_retrieval
    | contextual_prompt
    | llm
    | StrOutputParser()
)

In [15]:
# Initializing manual memory
chat_history = []

queries = [
    "What are the main architectural techniques the authors used to speed up training?",
    "Can you elaborate on the first one and why it outperformed traditional methods?",
    "How does this specific choice relate to the overall goal of the ImageNet competition?"
]

for q in queries:
    # Run the chain
    response = rag_chain.invoke({
        "question": q,
        "chat_history": chat_history
    })

    print(f"User: {q}")
    print(f"AI: {response}\n")
    print("-" * 30)

    chat_history.extend([
        HumanMessage(content=q),
        AIMessage(content=response)
    ])

User: What are the main architectural techniques the authors used to speed up training?
AI: The text does not explicitly mention the main architectural techniques used to speed up training. However, in the context of the AlexNet paper and typical deep learning architectures at that time, some common techniques used to prevent overfitting (not explicitly mentioned in the text) and potentially speed up training include:

1. **Local Response Normalization (LRN)**: a technique that normalizes the output of a convolutional layer to reduce the effect of overlapping regions.
2. **Dropout**: a regularization technique that randomly sets a fraction of hidden units to zero during training to prevent overfitting.
3. **ReLU (Rectified Linear Unit) activation function**: a non-linear activation function that outputs 0 for inputs less than 0 and the input value for inputs greater than or equal to 0. ReLU is often used in place of the traditional sigmoid or tanh activation function.
4. **Deep network